In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# =========================
# LOAD DATA
# =========================
train_ds = keras.utils.image_dataset_from_directory(
    r"F:\projects gam3a\Smart-Waste-Classification-NN-PROJECT-\data_processed\test",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    r"data_processed/val",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
np.save("classes.npy", class_names)   # ✅ important fix
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =========================
# MODEL
# =========================
base_model = keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    # ⚠️ KEEP normalization ONLY HERE
    layers.Rescaling(1./127.5, offset=-1),

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    ]
)

model.save("waste_classifier_v2.keras")
print("✅ Model saved")

# paula ---> we need to rise the epochs to 15 and make early stop


Found 382 files belonging to 6 classes.
Using 306 files for training.
Found 374 files belonging to 6 classes.
Using 74 files for validation.


9406464/9406464 [==============================] - 3s 0us/step
Epoch 1/10


10/10 [==============================] - 4s 307ms/step - loss: 2.1747 - accuracy: 0.1993 - val_loss: 1.6075 - val_accuracy: 0.4054
Epoch 2/10
10/10 [==============================] - 2s 228ms/step - loss: 1.6794 - accuracy: 0.3497 - val_loss: 1.2757 - val_accuracy: 0.5676
Epoch 3/10
10/10 [==============================] - 2s 234ms/step - loss: 1.3626 - accuracy: 0.4902 - val_loss: 1.1304 - val_accuracy: 0.5811
Epoch 4/10
10/10 [==============================] - 2s 243ms/step - loss: 1.2000 - accuracy: 0.5490 - val_loss: 1.0522 - val_accuracy: 0.6081
Epoch 5/10
10/10 [==============================] - 2s 243ms/step - loss: 0.9947 - accuracy: 0.6144 - val_loss: 0.9706 - val_accuracy: 0.6351
Epoch 6/10
10/10 [==============================] - 3s 256ms/step - loss: 0.9171 

In [2]:
import tensorflow as tf
import numpy as np

# =========================
# LOAD MODEL + CLASSES
# =========================
model = tf.keras.models.load_model("waste_classifier_v2.keras")
class_names = np.load("classes.npy", allow_pickle=True)

IMG_SIZE = (224, 224)

# =========================
# PREDICT FUNCTION
# =========================
def predict_image(img_path):

    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)

    # ⚠️ IMPORTANT: NO normalization here
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)

    class_index = np.argmax(preds)
    confidence = np.max(preds)

    print("Predicted Class:", class_names[class_index])
    print("Confidence:", float(confidence))


# =========================
# TEST
# =========================
predict_image(r"data_processed/test/metal/000095.jpg")

1/1 [==============================] - 0s 389ms/step
Predicted Class: metal
Confidence: 0.9117141366004944
